In [ ]:
# %pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [1]:
from langchain_core.documents import Document

In [2]:
sample_doc = Document(
    page_content="Hello World",
    metadata={"source": "https:/www.google.com"}
)

In [3]:
sample_doc

Document(metadata={'source': 'https:/www.google.com'}, page_content='Hello World')

In [4]:
type(sample_doc)

langchain_core.documents.base.Document

#### Load text and pdf data

In [ ]:
# Text data
# from langchain_community.document_loaders import TextLoader

# loader = TextLoader("data/Python.txt", encoding="utf-8")

# document = loader.load()
# document

[Document(metadata={'source': 'data/Python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

In [ ]:
# PDF data
# from langchain_community.document_loaders import PyPDFLoader

# pdf_loader = PyPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document

In [ ]:
# PDF data
# from langchain_community.document_loaders import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document

### Ingestion Pipeline

#### Document

In [16]:
# Data => Document
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [17]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []
    
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)
            
            pdf_loader = PyPDFLoader(pdf_path)
            doc = pdf_loader.load()
            
            all_docs.extend(doc)
            num_docs += 1
            
    print("total pdfs:", num_docs)
    print("total pages", len(all_docs))
    
    return all_docs

In [18]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages 36


In [20]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

#### Chunks

In [23]:
# chunks
# %pip install langchain_text_splitters 

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# chunk_size = maximum characters in each chunk, chunk_overlap = overlap of characters between 2 chunks
def split_docs(documents, chunk_size=500, chunk_overlap=50):
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )
    
    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [27]:
chunks = split_docs(all_pdf_documents)

In [28]:
len(chunks)

337

#### Embedding

In [30]:
from sentence_transformers import SentenceTransformer

In [33]:
class EmbeddingManager:
    def __init__(self, model_name = "all-MiniLM-L6-v2"):
        
        self.model_name = model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())
        
    def get_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embedding shape", embeddings.shape)
        return embeddings

In [34]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2
embedding dimensions= 384


C:\Users\user\AppData\Local\Temp\ipykernel_19360\3466090531.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


#### Vector Store

In [35]:
import chromadb
import uuid  # creates indexes for our individual documents

In [ ]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory # hard disk path where vector store is located
        self.collection = None
        self.client = None  # helps connecting others with vector store
        
        self._initialize_store()
        
    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True) # if vector store doesnot exist create it at persist directory path